## Task 2.2: Reproduction
* **Attempting to reproduce**: The hierarchical SVM with Orthogonal Transfer objective function (Section 3, Equation 4).
* **Evaluation metric**: Accuracy (0/1 Loss equivalent).

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# Load the realistic Wine data
df = pd.read_csv('partB/data/wine_dataset.csv')
X = df.drop('target', axis=1).values
y = df['target'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

class HierarchicalSVM(nn.Module):
    def __init__(self, num_nodes, num_features, K_matrix, C=0.5):
        super(HierarchicalSVM, self).__init__()
        self.W = nn.Parameter(torch.randn(num_nodes, num_features))
        self.K = torch.tensor(K_matrix, dtype=torch.float32)
        self.C = C

    def orthogonal_regularization(self):
        reg_loss = 0.0
        num_nodes = self.W.shape[0]
        for i in range(num_nodes):
            for j in range(num_nodes):
                if self.K[i, j] > 0:
                    dot_product = torch.abs(torch.dot(self.W[i], self.W[j]))
                    reg_loss += self.K[i, j] * dot_product
        return 0.5 * reg_loss

    def forward(self, x):
        return torch.matmul(x, self.W.T)

num_nodes = 5
num_features = 13 # Updated for Wine dataset
K_matrix = np.zeros((num_nodes, num_nodes))

# Enforce Ancestor relationships
K_matrix[1, 0] = K_matrix[0, 1] = 1.0
K_matrix[2, 0] = K_matrix[0, 2] = 1.0
K_matrix[3, 2] = K_matrix[2, 3] = 1.0
K_matrix[4, 2] = K_matrix[2, 4] = 1.0
for i in range(num_nodes): K_matrix[i, i] = 2.0

model = HierarchicalSVM(num_nodes, num_features, K_matrix, C=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop
epochs = 250
for epoch in range(epochs):
    optimizer.zero_grad()
    scores = model(X_train_t)

    cls_loss = 0.0
    for i in range(len(X_train_t)):
        label = y_train_t[i]
        s = scores[i]
        if label == 0:
            cls_loss += torch.relu(1.0 - (s[1] - s[2]))
        elif label == 1:
            cls_loss += torch.relu(1.0 - (s[2] - s[1]))
            cls_loss += torch.relu(1.0 - (s[3] - s[4]))
        elif label == 2:
            cls_loss += torch.relu(1.0 - (s[2] - s[1]))
            cls_loss += torch.relu(1.0 - (s[4] - s[3]))

    cls_loss = cls_loss / len(X_train_t)
    total_loss = cls_loss + model.C * model.orthogonal_regularization()
    total_loss.backward()
    optimizer.step()

def recursive_predict(model, x):
    model.eval()
    with torch.no_grad():
        scores = model(x.unsqueeze(0)).squeeze()
        if scores[1] > scores[2]: return 0
        else:
            if scores[3] > scores[4]: return 1
            else: return 2

predictions = [recursive_predict(model, x) for x in X_test_t]
acc = accuracy_score(y_test, predictions)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 90.74%
